# 12 — Architecture, Ports and Adapters, Dependency Injection

Separate domain rules from storage, network, and presentation layers. Learn why explicit dependencies, small interfaces, and transactional boundaries make code easier to test and evolve.

*Advanced · 90–150 min*

## You will learn

- Separate pure domain code from I/O adapters
- Create small protocol-based ports
- Inject repositories and time sources
- Identify boundaries and domain-specific errors

## Examples

### 1. Core domain rules

Domain logic should receive plain input and return plain results without opening files or making requests. The example models a reservation policy, allowing tests to explore conflicts without depending on a database.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Reservation:
    item_id: str
    user_id: str

def can_reserve(item_id, active):
    return item_id not in {reservation.item_id for reservation in active}

reservations = [Reservation("A", "u1")]
print(can_reserve("B", reservations), can_reserve("A", reservations))

**Design review:** Should a production reservation service scan all active reservations every time?

### 2. Ports and adapters

A port specifies required operations while adapters implement storage, network, or in-memory behavior. Protocol typing helps communicate the contract; the interpreter does not enforce the annotation automatically.

In [ ]:
from typing import Protocol

class ReservationRepo(Protocol):
    def exists(self, item_id: str) -> bool: ...
    def save(self, record: Reservation) -> None: ...

class InMemoryRepo:
    def __init__(self):
        self.data = {}
    def exists(self, item_id):
        return item_id in self.data
    def save(self, record):
        self.data[record.item_id] = record

def reserve(repo: ReservationRepo, record: Reservation):
    if repo.exists(record.item_id):
        raise ValueError("already reserved")
    repo.save(record)
    return record

repo = InMemoryRepo()
print(reserve(repo, Reservation("B", "u2")))

**Design review:** Why is the check-then-save implementation not atomic across concurrent callers?

### 3. Injecting time and randomness

Functions that read the real clock or random source can be hard to test. Inject a clock or random generator so tests control nondeterministic values and are not sensitive to wall-clock timing.

In [ ]:
from datetime import date

def membership_expired(expiry, today_fn):
    return expiry < today_fn()

fixed_today = lambda: date(2026, 11, 20)
print(membership_expired(date(2026, 11, 19), fixed_today))
print(membership_expired(date(2026, 11, 20), fixed_today))

**Design review:** Why should a real clock be timezone-aware when working with instants?

### 4. Errors at boundaries

Translate low-level I/O failures at the adapter boundary without hiding their cause. Domain code can raise meaningful exceptions that callers convert into HTTP statuses or CLI error messages.

In [ ]:
class ReservationConflict(Exception):
    pass

def reserve_or_conflict(repo, record):
    try:
        return reserve(repo, record)
    except ValueError as exc:
        raise ReservationConflict("item unavailable") from exc

try:
    reserve_or_conflict(repo, Reservation("B", "u3"))
except ReservationConflict as exc:
    print(type(exc).__name__, exc)

**Design review:** What kind of database constraint is needed to enforce uniqueness under concurrency?

## Exercises

**Challenge 1.** Design a storage Protocol with two implementations and the same behavior.

In [ ]:
# Challenge 1
# TODO: Write implementation and assertions.

**Challenge 2.** Refactor a service to accept a clock and test expiry boundaries.

In [ ]:
# Challenge 2
# TODO: Write implementation and assertions.

**Challenge 3.** Document what data should cross domain/adaptor boundaries.

In [ ]:
# Challenge 3
# TODO: Write implementation and assertions.

**Challenge 4.** Explain why a database transaction is necessary for check-then-insert races.

In [ ]:
# Challenge 4
# TODO: Write implementation and assertions.

- An in-memory duplicate check is not a replacement for a database uniqueness constraint.
- Avoid pushing HTTP or SQL-specific exceptions deep into domain logic.
- Too many abstraction layers can obscure small applications.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** In ports and adapters, the domain code should…

   - **a)** Depend on small interfaces (ports), not concrete I/O
   - **b)** Call the database directly
   - **c)** Read environment variables everywhere

**2.** Injecting a clock lets you…

   - **a)** Test time-dependent logic deterministically
   - **b)** Avoid datetime
   - **c)** Make code faster

**3.** A fake repository in tests is…

   - **a)** Production code
   - **b)** A simple in-memory implementation of the port
   - **c)** A mocked database server

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/12', {1: '73bb11f6f67a', 2: '99925a6649ad', 3: '944ad110d95b'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [13 — Security: Validation, Safe Parsing, Secrets and Dependencies](13_security_safe_parsing_secrets.ipynb)